# Phase 5 · probability-averaging ensemble
Reads `dev_probs.npy` / `test_probs.npy` written by each run notebook's cell 6, averages them, tunes thresholds on dev and writes two more submissions. CPU only. Scores are reported with per-statute thresholds estimated honestly (tuned on half of dev, scored on the other half), because choosing runs AND thresholds on the same dev data flatters every number.

In [ ]:
# 1 · Settings
from pathlib import Path
SEED = 17
COURSE_ROOT = Path('/content/drive/MyDrive/legal-llm-course'); PHASE_DIR = COURSE_ROOT/'Phase_5_LSI'; RUNS_ROOT = COURSE_ROOT/'runs/phase5_lsi'
BUNDLE_ZIP  = COURSE_ROOT/'phase2_lner_multimethod_bundle.zip'
RR_LIB, CJPE_LIB, LSI_LIB = COURSE_ROOT/'rr_lib.py', COURSE_ROOT/'cjpe_lib.py', COURSE_ROOT/'lsi_lib.py'
CANDIDATES = [RUNS_ROOT/f'NEW-{m}/full/new-{n}-full-00{i}' for i in (1, 2) for m, n in (('GQA', 'gqa'), ('ENC', 'enc'), ('HYB', 'hyb'))]
# add the older runs here to include them (every subset of RUN_DIRS is scored, so keep the list short): RUNS_ROOT/'MHA-prenorm/full/mha-full-001', ...
WEIGHTS = None            # e.g. [2, 1, 1]
OUT = RUNS_ROOT/'ensemble/ens-001'

In [ ]:
# 2 · Drive, helper bundle, tokenizer, device
from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, zipfile, hashlib, json, subprocess, importlib.util, random
LOCAL = Path('/content/phase3-rr')
if LOCAL.exists(): shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as z: z.extractall(LOCAL)
shutil.copy(RR_LIB, LOCAL/'rr_lib.py'); shutil.copy(CJPE_LIB, LOCAL/'cjpe_lib.py'); shutil.copy(LSI_LIB, LOCAL/'lsi_lib.py')
sys.path.insert(0, str(LOCAL))
need = [p for p in ('tokenizers','huggingface_hub','scikit-learn') if importlib.util.find_spec(p.replace('scikit-learn','sklearn')) is None]
if need: subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])
import torch, rr_lib as R, cjpe_lib as J, lsi_lib as LSI
from tokenizers import Tokenizer
from training_methods import resolve_snapshot_checkpoint
TOKENIZER_SHA256 = '906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab'
tp = LOCAL/'tokenizer/tokenizer.json'
assert hashlib.sha256(tp.read_bytes()).hexdigest() == TOKENIZER_SHA256, 'tokenizer differs from Phase 1/2'
TOK = Tokenizer.from_file(str(tp))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); torch.manual_seed(SEED)
import numpy as np
print('device', device, torch.cuda.get_device_name(0) if device.type=='cuda' else '')

In [ ]:
# 3 · Load every run's probabilities and check that ids line up with the official files
RUN_DIRS = [r for r in CANDIDATES if (r/'dev_probs.npy').exists()]      # runs that have finished (checked here, after Drive is mounted in cell 2)
print('ensembling', [r.name for r in RUN_DIRS])
for r in CANDIDATES:
    if r not in RUN_DIRS: print('  not found (no dev_probs.npy):', r, '| folder exists:', r.exists(), '| files:', sorted(x.name for x in r.glob('*'))[:12] if r.exists() else '-')
assert RUN_DIRS, 'no finished runs found under RUNS_ROOT: check the paths printed above (run folders are RUNS_ROOT/<PRESET>/full/<run id>)'
import itertools, pandas as pd
dev_docs = LSI.load_split(PHASE_DIR/'Train/dev.json', dedup=True); test_docs = LSI.load_split(PHASE_DIR/'Test/test.json', keep_labels=False)
Y = LSI.multi_hot(dev_docs); dev_ids = [d['id'] for d in dev_docs]; test_ids = [d['id'] for d in test_docs]
PD = [np.load(r/'dev_probs.npy') for r in RUN_DIRS]; PT = [np.load(r/'test_probs.npy') for r in RUN_DIRS]
for r in RUN_DIRS:
    assert json.loads((r/'dev_ids.json').read_text()) == dev_ids and json.loads((r/'test_ids.json').read_text()) == test_ids, f'{r.name}: ids differ from the official files'
w = np.array(WEIGHTS or [1.0]*len(RUN_DIRS)); w = w / w.sum()
def row(name, P):
    ev = LSI.evaluate_probs(Y, P); return {'members': name, 'macro_F1 (global thr)': round(ev['global_tuned+min_one']['macro_f1'], 4), 'macro_F1 (per-label, cross-fit)': round(ev['per_label_cross_fit+min_one']['macro_f1'], 4), 'micro_F1': round(ev['global_tuned+min_one']['micro_f1'], 4)}
rows = [row(r.name, p) for r, p in zip(RUN_DIRS, PD)]
for n in range(2, len(RUN_DIRS) + 1):
    for sub in itertools.combinations(range(len(RUN_DIRS)), n): rows.append(row('+'.join(RUN_DIRS[i].name for i in sub), np.mean([PD[i] for i in sub], 0)))
display(pd.DataFrame(rows).sort_values('macro_F1 (per-label, cross-fit)', ascending=False))

In [ ]:
# 4 · Write the ensemble submissions (all runs in RUN_DIRS, weights as set above)
P_dev = sum(wi * p for wi, p in zip(w, PD)); P_test = sum(wi * p for wi, p in zip(w, PT))
g, _ = LSI.tune_global(Y, P_dev); th = LSI.tune_per_label(Y, P_dev)
for tag, thr in (('submission_global', g), ('submission_perlabel', th)):
    pred = LSI.decide(P_test, thr, True); path = LSI.write_submission(P_test, test_ids, pred, OUT/tag)
    print(tag, '| statutes per document', round(float(pred.sum(1).mean()), 2), '->', path)
print('dev macro-F1 of this ensemble (global threshold, in-sample):', round(LSI.scores(Y, LSI.decide(P_dev, g, True))['macro_f1'], 4))

In [ ]:
# 5 · TF-IDF + linear model as one more ensemble member (CPU; the first run takes a while, then it is cached on Drive)
USE_TFIDF = True
TF_DEV = TF_TEST = None
if USE_TFIDF:
    import time
    TF_CACHE = RUNS_ROOT/'tfidf/probs.npz'
    z = np.load(TF_CACHE) if TF_CACHE.exists() else None
    if z is not None and list(z['dev_ids']) == dev_ids and list(z['test_ids']) == test_ids:
        TF_DEV, TF_TEST = z['dev'], z['test']; print('loaded cached TF-IDF probabilities:', str(z['name']) if 'name' in z.files else 'first baseline version')
    else:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.multiclass import OneVsRestClassifier
        from sklearn.svm import LinearSVC
        t0 = time.time()
        tr_docs = LSI.load_split(PHASE_DIR/'Train/train.json', dedup=True); ytr = LSI.multi_hot(tr_docs)
        cut = lambda ds: [d['text'][:60000] for d in ds]
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200000, sublinear_tf=True, dtype=np.float32)
        A = vec.fit_transform(cut(tr_docs)); print('tf-idf matrix', A.shape, round(time.time() - t0), 's')
        clf = OneVsRestClassifier(LinearSVC(C=0.1), n_jobs=-1).fit(A, ytr)
        sig = lambda S: (1 / (1 + np.exp(-4 * S))).astype(np.float32)        # squash scores to (0,1), same as the baseline notebook
        TF_DEV, TF_TEST = sig(clf.decision_function(vec.transform(cut(dev_docs)))), sig(clf.decision_function(vec.transform(cut(test_docs))))
        TF_CACHE.parent.mkdir(parents=True, exist_ok=True)
        np.savez(TF_CACHE, dev=TF_DEV, test=TF_TEST, dev_ids=np.array(dev_ids), test_ids=np.array(test_ids)); print('done in', round(time.time() - t0), 's')
    ev = LSI.evaluate_probs(Y, TF_DEV)
    print('TF-IDF alone on dev: macro-F1 global', round(ev['global_tuned+min_one']['macro_f1'], 4), '| per-label cross-fit', round(ev['per_label_cross_fit+min_one']['macro_f1'], 4))

In [ ]:
# 6 · Two cheap improvements, both chosen by cross-fitting on dev: (a) how much TF-IDF to mix in, (b) how far to shrink each statute's threshold toward the global one
# threshold for statute k = lam_k * (its own tuned threshold) + (1 - lam_k) * global,  lam_k = n_k / (n_k + m),  n_k = positives of that statute in the tuning half
# m = 0 is plain per-label thresholds, m = inf is one global threshold. Rare statutes have noisy own-thresholds, so they lean on the global value.
WS = (0.0, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0) if TF_DEV is not None else (0.0,)      # 1.0 = TF-IDF alone; if the best weight is at the top of this list, extend it
MS = (0, 30, 100, 300, float('inf'))
def mix(Pn, Pt, wt): return Pn if wt == 0 else (1 - wt) * Pn + wt * Pt
def thresholds(f, n, m):
    g = LSI.GRID[int(np.argmax(f.mean(1)))]; t = LSI.GRID[np.argmax(f, 0)]
    lam = np.ones_like(t) if m == 0 else (np.zeros_like(t) if not np.isfinite(m) else n / (n + m))
    return lam * t + (1 - lam) * g
res = {}
for wt in WS:
    P = mix(P_dev, TF_DEV, wt)
    for sd in range(5):                                                        # five random 2-fold splits of dev
        halves = np.array_split(np.random.RandomState(sd).permutation(len(Y)), 2)
        for a, b in ((0, 1), (1, 0)):
            tr, te = halves[a], halves[b]; f = LSI._f1_by_threshold(Y[tr], P[tr]); n = Y[tr].sum(0)
            for m in MS: res.setdefault((wt, m), []).append(LSI.scores(Y[te], LSI.decide(P[te], thresholds(f, n, m), True))['macro_f1'])
tab = pd.DataFrame({(('inf' if not np.isfinite(m) else m)): {wt: np.mean(res[(wt, m)]) for wt in WS} for m in MS}).round(4)
tab.index.name = 'TF-IDF weight'; tab.columns.name = 'shrinkage m (0 = per-label, inf = global)'
display(tab)
best = max(res, key=lambda k: np.mean(res[k])); base = np.mean(res[(0.0, 0)])
print('best (TF-IDF weight, m):', best, '| cross-fit macro-F1', round(float(np.mean(res[best])), 4), '| plain neural ensemble with per-label thresholds', round(float(base), 4))

In [ ]:
# 7 · Write the tuned ensemble submission (settings from cell 6; thresholds tuned on all of dev)
wt, m = best
P_d, P_t = mix(P_dev, TF_DEV, wt), mix(P_test, TF_TEST, wt)
thr = thresholds(LSI._f1_by_threshold(Y, P_d), Y.sum(0), m)
pred = LSI.decide(P_t, thr, True); OUT2 = RUNS_ROOT/'ensemble/ens-002'
path = LSI.write_submission(P_t, test_ids, pred, OUT2/'submission_tuned')
print('TF-IDF weight', wt, '| m', m, '| statutes per document', round(float(pred.sum(1).mean()), 2), '->', path)
print('dev macro-F1 with these thresholds (in-sample, flattering):', round(LSI.scores(Y, LSI.decide(P_d, thr, True))['macro_f1'], 4))